In [1]:
# 1. Standard Imports
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from dbconfig import engine

print("Environment Ready.")

Environment Ready.


In [74]:
%%sql
with total_matches as
(select team1 as team, count(distinct match_id) as c
from ipl_master_view
group by team1

union all

select team2 as team, count(distinct match_id) as c
from ipl_master_view
group by team2)

select team, sum(c) as match_count
from total_matches
group by team
order by match_count desc;



,team,match_count
0,Mumbai Indians,261
1,Sunrisers Hyderabad,257
2,Royal Challengers Bengaluru,255
3,Delhi Capitals,252
4,Kolkata Knight Riders,251
5,Punjab Kings,246
6,Chennai Super Kings,238
7,Rajasthan Royals,221
8,Pune Warriors,46
9,Gujarat Titans,45


In [75]:
%%sql
select winner, count(distinct match_id) as win_count
from ipl_master_view
where winner is not null
group by winner
order by win_count desc;

,winner,win_count
0,Mumbai Indians,144
1,Chennai Super Kings,138
2,Kolkata Knight Riders,131
3,Royal Challengers Bengaluru,123
4,Sunrisers Hyderabad,117
5,Delhi Capitals,115
6,Punjab Kings,112
7,Rajasthan Royals,112
8,Gujarat Titans,28
9,Lucknow Super Giants,24


In [57]:
%%sql
select distinct result
from ipl_master_view;

,result
0,no result
1,runs
2,tie
3,wickets


In [63]:
%%sql
with runs_margin as (
select avg(result_margin) as r_m
from ipl_master_view
where result = 'runs'
and winner = 'Mumbai Indians'),

wickets_margin as (
select avg(result_margin) as w_m
from ipl_master_view
where result = 'wickets'
and winner = 'Mumbai Indians')

select r_m as runs_margin, w_m as wickets_margin
from runs_margin, wickets_margin;

,runs_margin,wickets_margin
0,32.241587,6.131718


In [88]:
result = match_count.merge(win_count, left_on='team', right_on='winner', how='left')
result = result.drop(columns=['winner'])
result['win_pct'] = (result['win_count'] / result['match_count'] * 100.0).round(2)
result

,team,match_count,win_count,win_pct
0,Mumbai Indians,261,144,55.17
1,Sunrisers Hyderabad,257,117,45.53
2,Royal Challengers Bengaluru,255,123,48.24
3,Delhi Capitals,252,115,45.63
4,Kolkata Knight Riders,251,131,52.19
5,Punjab Kings,246,112,45.53
6,Chennai Super Kings,238,138,57.98
7,Rajasthan Royals,221,112,50.68
8,Pune Warriors,46,12,26.09
9,Gujarat Titans,45,28,62.22


In [85]:
win_pct_top_3_played = result[result['team'].isin(['Mumbai Indians', 'Chennai Super Kings', 'Royal Challengers Bengaluru'])].sort_values(by = 'win_pct', ascending = False)
win_pct_top_3_played

,team,match_count,win_count,win_pct
6,Chennai Super Kings,238,138,57.98
0,Mumbai Indians,261,144,55.17
2,Royal Challengers Bengaluru,255,123,48.24


In [11]:
%%sql
select count(*)
from matches
where toss_winner = winner

union all

select count(*)
from matches
where toss_winner != winner

union all

select count(*)
from matches;

,count
0,554
1,536
2,1095


In [7]:
%%sql
select toss_decision, count(*)
from matches
where toss_winner = winner
group by toss_decision;

,toss_decision,count
0,bat,177
1,field,377


In [10]:
%%sql
select
    toss_decision,
    count(*) as total_matches
from matches
group by toss_decision;

,toss_decision,total_matches
0,bat,391
1,field,704


In [29]:
%%sql
select venue,
       count(distinct id) as match_count,
       sum(case when team1 = winner then 1 else 0 end) as team1_wins,
       round(100.0 * sum(case when team1 = winner then 1 else 0 end) / count(distinct id), 2) as team1_win_pct
from matches
group by venue
order by match_count desc
limit 5;

,venue,match_count,team1_wins,team1_win_pct
0,Wankhede Stadium,118,59,50.00
1,M Chinnaswamy Stadium,94,46,48.94
2,Eden Gardens,93,53,56.99
3,MA Chidambaram Stadium,85,52,61.18
4,"Rajiv Gandhi International Stadium, Uppal",62,31,50.00


In [26]:
%%sql
select
    extract(year from to_date(date, 'YYYY-MM-DD')) as season,
    count(distinct id) as match_count,
    sum(case when winner = 'Mumbai Indians' then 1 else 0 end) as win_count,
    round(100.0 * sum(case when winner = 'Mumbai Indians' then 1 else 0 end)/ count(distinct id), 2) as win_pct
from matches
where team1 = 'Mumbai Indians' or team2 = 'Mumbai Indians'
group by extract(year from to_date(date, 'YYYY-MM-DD'))
order by season;

,season,match_count,win_count,win_pct
0,2008,14,7,50.00
1,2009,13,5,38.46
2,2010,16,11,68.75
3,2011,16,10,62.50
4,2012,17,10,58.82
5,2013,19,13,68.42
6,2014,15,7,46.67
7,2015,16,10,62.50
8,2016,14,7,50.00
9,2017,17,12,70.59


In [33]:
import polars as pl

#mi_ipl = pl.from_pandas(mi_ipl)
mi_ipl = mi_ipl.with_columns(
    pl.col('win_pct')
    .rolling_mean(window_size=4)
    .alias('rolling_3yr')
)

mi_ipl

season,match_count,win_count,win_pct,rolling_3yr
i64,i64,i64,f64,f64
2008,14,7,50.0,null
2009,13,5,38.46,null
2010,16,11,68.75,null
2011,16,10,62.5,54.9275
2012,17,10,58.82,57.1325
…,…,…,…,…
2020,16,11,68.75,62.7375
2021,14,7,50.0,57.59
2022,14,4,28.57,54.0175


In [46]:
from scipy.stats import linregress
import pandas as pd

mi_post_2021 = mi_ipl.filter(pl.col('season') >= 2021)

slope, intercept, r_value, p_value, std_err = linregress(
    mi_post_2021['season'],
    mi_post_2021['win_pct']
)

print(f"Slope: {slope:.2f}% per year")
print(f"Intercept: {intercept:.2f}%")
print(f"R²: {r_value**2:.3f}")
print(f"p-value: {p_value:.4f}")

Slope: -3.66% per year
Intercept: 7445.22%
R²: 0.108
p-value: 0.6719


In [42]:
%%sql
select
    sum(case when winner = 'Chennai Super Kings' then 1 else 0 end) as win_count_csk,
    sum(case when winner = 'Mumbai Indians' then 1 else 0 end) as win_count_mi

from matches
where team1 in ('Chennai Super Kings', 'Mumbai Indians') and team2 in ('Chennai Super Kings', 'Mumbai Indians');

,win_count_csk,win_count_mi
0,17,20


In [24]:
%%sql
select
    case when(extract(year from to_date(date, 'YYYY-MM-DD'))) < 2020 then 'pre_2020' else 'post_2020' end as season,
    count(distinct id) as match_count,
    sum(case when winner = 'Mumbai Indians' then 1 else 0 end) as win_count,
    round(100.0 * sum(case when winner = 'Mumbai Indians' then 1 else 0 end)/ count(distinct id), 2) as win_pct
from matches
where team1 = 'Mumbai Indians' or team2 = 'Mumbai Indians'
group by case when(extract(year from to_date(date, 'YYYY-MM-DD'))) < 2020 then 'pre_2020' else 'post_2020' end
order by season;

,season,match_count,win_count,win_pct
0,post_2020,74,35,47.30
1,pre_2020,187,109,58.29


In [2]:
%%sql
select
    case when extract(year from to_date(date, 'YYYY-MM-DD')) between 2013 and 2019 then 'Pre-2020'
         when extract(year from to_date(date, 'YYYY-MM-DD')) between 2020 and 2024 then 'Post-2020' end as period,
    case when winner = 'Chennai Super Kings' then 'Win' else 'Loss' end as outcome,
    count(distinct id) as count
from matches
where team1 = 'Chennai Super Kings' or team2 = 'Chennai Super Kings'
group by period, outcome
order by period, outcome;

,period,outcome,count
0,Post-2020,Loss,36
1,Post-2020,Win,38
2,Pre-2020,Loss,31
3,Pre-2020,Win,53
4,NaN,Loss,33
5,NaN,Win,47


In [12]:
from scipy.stats import chi2_contingency
import numpy as np

contingency = np.array([
    [177, 214],  # Bat: won, lost
    [377, 327]   # Field: won, lost
])

chi2, p, dof, expected = chi2_contingency(contingency)

print(f"Chi-Squared: {chi2}")
print(f"p-value: {p}")
print(f"Degrees of Freedom: {dof}")

Chi-Squared: 6.571677733078204
p-value: 0.01036142309365386
Degrees of Freedom: 1


In [5]:
chi2 = 1.7668181959226748
n = 158  # 164 + 74
k = 2

cramers_v = np.sqrt(chi2 / (n * (k - 1)))
print(f"Cramér's V: {cramers_v}")

Cramér's V: 0.10574683751810368


In [13]:
%%sql
select
    result,
    count(*) as matches
from matches
where result in ('runs', 'wickets')
group by result;

,result,matches
0,runs,498
1,wickets,578


In [23]:
%%sql
with team_matches as (
    select
        team1 as team,
        extract(year from to_date(date, 'YYYY-MM-DD')) as season,
        case when winner = team1 then 1 else 0 end as win
    from matches

    union all

    select
        team2 as team,
        extract(year from to_date(date, 'YYYY-MM-DD')) as season,
        case when winner = team2 then 1 else 0 end as win
    from matches
)


select
    team,
    season,
    count(*) as match_count,
    sum(win) as win_count,
    round(100.0 * avg(win), 2) as win_pct
from team_matches
group by team, season
order by team, season;

,team,season,match_count,win_count,win_pct
0,Chennai Super Kings,2008,16,9,56.25
1,Chennai Super Kings,2009,14,8,57.14
2,Chennai Super Kings,2010,16,9,56.25
3,Chennai Super Kings,2011,16,11,68.75
4,Chennai Super Kings,2012,18,10,55.56
...,...,...,...,...,...
141,Sunrisers Hyderabad,2020,16,8,50.00
142,Sunrisers Hyderabad,2021,14,3,21.43
143,Sunrisers Hyderabad,2022,14,6,42.86
144,Sunrisers Hyderabad,2023,14,4,28.57
